# Movie Recommendation System 

This project implements a movie recommendation system that combines Python with Prolog-based symbolic reasoning.

The system first identifies similar movies based on shared genres using a Prolog knowledge base. It then incorporates user ratings to generate personalized recommendations and evaluates performance using Precision, Recall, and F1-score.

In [10]:
import pandas as pd
from pyswip import Prolog

data = pd.read_csv("data/movies_metadata.csv")

# Replace missing values only in text columns
text_columns = data.select_dtypes(include=["object", "string"]).columns
data[text_columns] = data[text_columns].fillna("UNK")

# Preview the first five rows
data.head()

,Unnamed: 0,budget,genres,homepage,id,plot_keywords,language,original_title,overview,popularity,...,tagline,movie_title,vote_average,num_voted_users,title_year,country,director_name,actor_1_name,actor_2_name,actor_3_name
0,0,237000000,Action|Adventure|Fantasy|Science Fiction,http://www.avatarmovie.com/,19995,culture clash|future|space war|space colony|so...,English,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,...,Enter the World of Pandora.,Avatar,7.2,11800,2009.0,United States of America,James Cameron,Zoe Saldana,Sigourney Weaver,Stephen Lang
1,1,300000000,Adventure|Fantasy|Action,http://disney.go.com/disneypictures/pirates/,285,ocean|drug abuse|exotic island|east india trad...,English,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,...,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,2007.0,United States of America,Gore Verbinski,Orlando Bloom,Keira Knightley,Stellan Skarsgård
2,2,245000000,Action|Adventure|Crime,http://www.sonypictures.com/movies/spectre/,206647,spy|based on novel|secret agent|sequel|mi6|bri...,Français,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,...,A Plan No One Escapes,Spectre,6.3,4466,2015.0,United Kingdom,Sam Mendes,Christoph Waltz,Léa Seydoux,Ralph Fiennes
3,3,250000000,Action|Crime|Drama|Thriller,http://www.thedarkknightrises.com/,49026,dc comics|crime fighter|terrorist|secret ident...,English,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,...,The Legend Ends,The Dark Knight Rises,7.6,9106,2012.0,United States of America,Christopher Nolan,Michael Caine,Gary Oldman,Anne Hathaway
4,4,260000000,Action|Adventure|Science Fiction,http://movies.disney.com/john-carter,49529,based on novel|mars|medallion|space travel|pri...,English,John Carter,"John Carter is a war-weary, former military ca...",43.926995,...,"Lost in our world, found in another.",John Carter,6.1,2124,2012.0,United States of America,Andrew Stanton,Lynn Collins,Samantha Morton,Willem Dafoe


In [11]:
def clean_text(text):
  text = text.replace(u'\xa0', u'')
  text = text.replace(u"'", u'')
  return text

In [12]:
# Initialize the Prolog knowledge base
prolog = Prolog()

# Create Prolog facts for each movie and its genres
literals = []
movie_score = {}

for row in data.itertuples(index=True, name="Pandas"):
    movie_title = clean_text(getattr(row, "movie_title"))

    for genre in getattr(row, "genres").split("|"):
        literals.append('genre("' + movie_title + '","' + genre + '")')

# Sort the facts before adding them to the knowledge base
literals.sort()

for literal in literals:
    prolog.assertz(literal)

# Load additional Prolog rules from the external knowledge base
prolog.consult("db.pl")

# Content-Based Recommendation System

This section implements a content-based movie recommendation system using the Prolog knowledge base created above.

Movie similarity is determined through Prolog rules based on shared movie characteristics. The `find_sim_1` rule identifies movies that share at least one genre, providing a simple similarity criterion for generating recommendations.

In [13]:
def simple_recommender(movie):
    recommendations = set()

    query = prolog.query(f'find_sim_1("{movie}", M)')

    for solution in query:
        recommendation = solution["M"]

        if isinstance(recommendation, bytes):
            recommendation = recommendation.decode("utf-8")

        recommendations.add(recommendation)

    query.close()

    return recommendations

In [14]:
list(simple_recommender('Avatar'))[:5]

['Supercross',
 'Fled',
 'The Call of Cthulhu',
 'Harold & Kumar Go to White Castle',
 'Mad Money']

# Personalized Recommendation System

This section extends the content-based recommender by incorporating user ratings.

The system uses the training ratings to assign scores to movies based on user preferences and the similarity relationships identified by the Prolog knowledge base. These learned scores are then used to predict user preferences on the test set.

Performance is evaluated using Precision, Recall, and F1-score.

In [15]:
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np
import random

# Ensure reproducible results
random.seed(42)

# Convert user ratings into preference weights
rating_weights = {
    0: -1,
    1: -0.5,
    2: 0,
    3: 0,
    4: 0.5,
    5: 1
}

# Weight assigned to the similarity level used by simple_recommender
score_weights = {i: i + 1 for i in range(1)}


def train_recommender(
    ratings,
    rating_weights,
    score_weights,
    number_of_movies=10
):
    """
    Train the recommendation system using a subset of user ratings.

    Parameters
    ----------
    ratings : pandas.DataFrame
        User ratings used for training.
    rating_weights : dict
        Mapping from rating values to preference weights.
    score_weights : dict
        Weights associated with movie similarity levels.
    number_of_movies : int
        Number of ratings to sample for training.
        Use -1 to train on the complete dataset.

    Returns
    -------
    dict
        Predicted preference score for each candidate movie.
    """

    if number_of_movies > len(ratings):
        number_of_movies = len(ratings)

    if number_of_movies != -1:
        indexes = random.sample(range(len(ratings)), number_of_movies)
        ratings = ratings.iloc[indexes]

    movie_score = {}

    for row in tqdm(ratings.itertuples(index=True, name="Pandas")):
        movie = clean_text(getattr(row, "movie_title"))
        rating = getattr(row, "rating")

        similar_movies = simple_recommender(movie)

        for similar_movie in similar_movies:
            score = rating_weights[int(rating)] * score_weights[0]

            if similar_movie not in movie_score:
                movie_score[similar_movie] = score
            else:
                movie_score[similar_movie] += score

    return movie_score


def predict_example(ratings, movie_score):
    """
    Predict whether each movie should be recommended.

    A movie is predicted as recommended when its learned score is
    greater than zero. Ratings above 3 are treated as positive
    preferences in the ground truth.
    """

    real = []
    pred = []

    for row in ratings.itertuples(index=True, name="Pandas"):
        movie = clean_text(getattr(row, "movie_title"))
        rating = getattr(row, "rating")

        if movie in movie_score:
            pred.append(int(movie_score[movie] > 0))
        else:
            pred.append(0)

        real.append(int(rating > 3))

    return real, pred


def get_metrics(real, pred):
    """Calculate precision, recall, and F1-score."""

    return {
        "precision": precision_score(real, pred),
        "recall": recall_score(real, pred),
        "f1": f1_score(real, pred)
    }

In [16]:
# Load training and test ratings
train_ratings = pd.read_csv("data/train_ratings.csv")
test_ratings = pd.read_csv("data/test_ratings.csv")

print(f"Training ratings: {len(train_ratings)}")
print(f"Test ratings: {len(test_ratings)}")

Training ratings: 100
Test ratings: 141


In [17]:
# Evaluate the recommender across multiple random training samples
metrics = []

for _ in range(10):
    movie_score = train_recommender(
        train_ratings,
        rating_weights,
        score_weights,
        number_of_movies=10
    )

    real, pred = predict_example(test_ratings, movie_score)
    metrics.append(get_metrics(real, pred))


# Report average performance across all runs
for metric in metrics[0].keys():
    mean_score = np.mean([m[metric] for m in metrics])
    print(f"{metric}: {mean_score:.3f}")

10it [00:03,  3.14it/s]
10it [00:02,  3.52it/s]
10it [00:02,  4.41it/s]
10it [00:02,  4.91it/s]
10it [00:02,  4.35it/s]
10it [00:02,  4.49it/s]
10it [00:02,  3.98it/s]
10it [00:02,  4.05it/s]
10it [00:02,  4.19it/s]
10it [00:02,  4.07it/s]

precision: 0.517
recall: 0.742
f1: 0.594


In [18]:
# Evaluate the recommender using 30 randomly sampled training ratings
metrics = []

for _ in range(10):
    movie_score = train_recommender(
        train_ratings,
        rating_weights,
        score_weights,
        number_of_movies=30
    )

    real, pred = predict_example(test_ratings, movie_score)
    metrics.append(get_metrics(real, pred))

# Report average performance across all runs
for metric in metrics[0].keys():
    mean_score = np.mean([m[metric] for m in metrics])
    print(f"{metric}: {mean_score:.3f}")

30it [00:07,  4.24it/s]
30it [00:06,  4.40it/s]
30it [00:07,  4.21it/s]
30it [00:07,  4.04it/s]
30it [00:06,  4.43it/s]
30it [00:06,  4.83it/s]
30it [00:07,  3.90it/s]
30it [00:05,  5.08it/s]
30it [00:06,  4.30it/s]
30it [00:07,  4.20it/s]

precision: 0.513
recall: 0.936
f1: 0.662


## Conclusion

This project demonstrates a hybrid recommendation approach that combines Prolog-based symbolic reasoning with user rating information to generate personalized movie recommendations.

Using 10 training ratings, the recommender achieved a mean Precision of 0.517, Recall of 0.742, and F1-score of 0.594. When the training sample was increased to 30 ratings, Recall increased to 0.936 and the F1-score to 0.662, while Precision remained relatively stable at 0.513.

These results suggest that using more user-rating information improved the system's ability to identify movies associated with positive user preferences, particularly in terms of Recall. However, the relatively lower Precision indicates that the system also produced a considerable number of false-positive recommendations.

Overall, the project illustrates how symbolic knowledge representation and simple preference-based scoring can be combined to build an interpretable recommendation system. Future improvements could incorporate additional movie features and more advanced recommendation techniques to improve recommendation precision.